# Comunidades propagadoras: heurística GRASP

Este notebook acompanha a nova etapa do projeto. O `main.ipynb` permanece como baseline congelado. A implementação reutilizável está em `community_heuristic.py`.

Nesta primeira seção validamos a rede, os parâmetros de coesão e as duas variantes em um recorte pequeno. A avaliação final usará a alcançabilidade determinística exata e comparará alcance externo e eficiência com as comunidades Leiden.

In [ ]:
import csv
import time

import networkx as nx

from community_heuristic import (
    ExactReach, MIAInfluence, TopologicalScore,
    cpm_score, load_baseline_graph, run_grasp,
)

graph = load_baseline_graph()
print(f'Rede do baseline: {len(graph):,} vértices e {graph.number_of_edges():,} arcos')

## Coesão CPM

Para a comunidade $C$, usamos $e(C)-\gamma |C|(|C|-1)\geq0$, equivalente a exigir densidade interna dirigida de pelo menos $\gamma$. A condição é verificada desde o primeiro par.

In [ ]:
with open('community_results/baseline/baseline_communities.csv', newline='', encoding='utf-8') as handle:
    baseline_rows = list(csv.DictReader(handle))
densities = sorted(
    (int(row['internal_influence_edges']) + int(row['internal_doctoral_supervision_edges']))
    / (int(row['community_size']) * (int(row['community_size']) - 1))
    for row in baseline_rows
)
gammas = (densities[23], (densities[47] + densities[48]) / 2)
print('Valores de gamma:', gammas)
assert len(baseline_rows) == 96

## Viabilidade do MIA em $p=1$

Sem poda por probabilidade, cada arborescência contém todos os ancestrais alcançáveis do alvo. Estimamos o número total de entradas antes de materializá-las. O limite acordado é de 4 GiB estimados ou 30 minutos de preparação; se for excedido, a comparação entre variantes será feita no mesmo recorte reproduzível.

In [ ]:
start = time.monotonic()
exact = ExactReach(graph)
pair_count = exact.reachable_pair_count()
estimated_gib = pair_count * 200 / 2**30
print(f'{pair_count:,} entradas; estimativa conservadora: {estimated_gib:.2f} GiB')
print(f'Preparação da alcançabilidade: {time.monotonic() - start:.1f} s')

## Ensaio inicial das duas variantes

O recorte abaixo usa busca em largura na projeção não dirigida, iniciada no vértice de maior grau de saída. Ele serve para verificar a execução, não para relatar resultados finais da rede completa.

In [ ]:
hub = min(graph, key=lambda node: (-graph.out_degree(node), str(node)))
pilot_nodes = list(nx.bfs_tree(graph.to_undirected(), hub))[:100]
pilot = graph.subgraph(pilot_nodes).copy()
pilot_exact = ExactReach(pilot)
pilot_mia = MIAInfluence(pilot, theta=1.0, default_probability=1.0)
assert all(abs(pilot_mia.total({node}) - pilot_exact.total({node})) < 1e-9 for node in pilot)
print(f'Recorte piloto: {len(pilot)} vértices, {pilot.number_of_edges()} arcos')
for gamma in gammas:
    for variant, scorer in [('topologica', TopologicalScore(pilot)), ('MIA', pilot_mia.external)]:
        solutions, elite = run_grasp(pilot, scorer, gamma, run_seeds=range(3))
        best = max(elite, key=lambda item: pilot_exact.external(item.nodes), default=None)
        print(gamma, variant, 'soluções:', len(solutions), 'elite:', len(elite),
              'melhor alcance externo:', pilot_exact.external(best.nodes) if best else None)